# Ingestão Bronze

Lê o arquivo bruto `vb_matches.csv` do Volume `bronze.raw_files` e grava a tabela Delta `bronze.vb_matches` sem nenhuma transformação.

**Fonte do Dataset:** TidyTuesday 2020-05-19 (`rfordatascience/tidytuesday`), espelhado no Kaggle (`jessemostipak/beach-volleyball`). Download em 16/09/2026, upload manual pelo Catalog Explorer.

## 1. Definição de caminhos da ingestão


In [0]:
from pyspark.sql import functions as F

CATALOGO = "workspace"
VOLUME_PATH = f"/Volumes/{CATALOGO}/bronze/raw_files"
ARQUIVO = "vb_matches.csv"
TABELA_DESTINO = f"{CATALOGO}.bronze.vb_matches"

display(dbutils.fs.ls(VOLUME_PATH))

## 2. Leitura bruta do CSV


In [0]:
df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")  # Deixa tudo como string
    .option("escape", '"')           # Aspas dentro de campo vêm como "" (padrão CSV)
    .load(f"{VOLUME_PATH}/{ARQUIVO}")
)

print(f"linhas: {df_raw.count()} | colunas: {len(df_raw.columns)}")
df_raw.printSchema()

## 3. Gravação da tabela bronze

In [0]:

df_bronze = (
    # Adicionando metadados para rastreabilidade
    df_raw
    .withColumn("_ingestao_ts", F.current_timestamp())
    .withColumn("_arquivo_origem", F.lit(f"{VOLUME_PATH}/{ARQUIVO}"))
    .withColumn("_camada", F.lit("bronze"))
)
(
    # Gravação em Delta 
    df_bronze.write.format("delta")
    .mode("overwrite")  # FULL LOAD
    .option("overwriteSchema", "true") 
    .saveAsTable(TABELA_DESTINO)
)

print(f"gravado em {TABELA_DESTINO}")

## 4. Validação pós-carga

In [0]:
df_tab = spark.table(TABELA_DESTINO)

assert df_tab.count() == df_raw.count(), "tabela e arquivo têm contagens diferentes"
assert set(df_raw.columns) <= set(df_tab.columns), "coluna do arquivo ausente na tabela"

print(f"{df_tab.count():,} linhas e {len(df_raw.columns)} colunas do arquivo preservadas em {TABELA_DESTINO}")